In [217]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from time import sleep
import logging
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(levelname)s - %(message)s',datefmt='%H:%M:%S',force=True)

In [218]:
driver = webdriver.Chrome()
logging.info('Браузер Selenium успешно запущен')
driver.get("https://store.steampowered.com/search/?ndl=1")
logging.info('Открыта страница поиска Steam')

22:13:40 - INFO - Браузер Selenium успешно запущен
22:13:40 - INFO - Открыта страница поиска Steam


In [219]:
def scrolldown(driver, vniz): #крч функция, которая мотает страницу вниз, чтобы больше игр прогрузилось
    logging.info('Начало прокрутки страницы')
    for i in range(vniz):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") #JavaScript-код для браузера, который обращается к DOM страницы
        #https://developer.mozilla.org/en-US/docs/Web/API/Window/scrollTo?.com
        #https://developer.mozilla.org/en-US/docs/Web/API/Document/body?.com
        #https://developer.mozilla.org/en-US/docs/Web/API/Element/scrollHeight?.com
        logging.info(f'Скролл номер {i+1}')
        sleep(2)
    logging.info('Прокрутка завершена')

In [220]:
scrolldown(driver, 5)

22:13:45 - INFO - Начало прокрутки страницы
22:13:45 - INFO - Скролл номер 1
22:13:47 - INFO - Скролл номер 2
22:13:49 - INFO - Скролл номер 3
22:13:51 - INFO - Скролл номер 4
22:13:53 - INFO - Скролл номер 5
22:13:55 - INFO - Прокрутка завершена


In [221]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
urls = []

for link in soup.find_all('a'):
    if '/app' in link.get('href') and link.get('href').startswith('https://'):
        urls.append(link.get('href'))
        
logging.info(f'Собрано ссылок на игры: {len(urls)}')

22:13:57 - INFO - Собрано ссылок на игры: 300


In [222]:
url0 = urls[3]

driver.get(url0)
html = driver.page_source
soup0 = BeautifulSoup(html, 'html.parser')
print(soup0.prettify())

<html class="responsive DesktopUI" lang="en">
 <head>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="#171a21" name="theme-color"/>
  <title>
   Slay the Spire 2 on Steam
  </title>
  <link href="/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/motiva_sans.css?v=YzJgj1FjzW34&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/shared_global.css?v=qbbUOQL2SbhT&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/buttons.css?v=BZhNEtESfYSJ&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/css/v6/store.css?v=13CjnPuGjLMj&amp;l=english&amp;_cdn=fastly" rel="stylesheet" typ

Для работы с BeautifulSoup была использована информация из источника: https://habr.com/ru/articles/986284/

In [223]:
name = soup0.find_all('div', {'class': 'apphub_AppName'})[0].text
print(name)

Slay the Spire 2


In [224]:
rating = soup0.find_all('span', {'class': 'game_review_summary positive'})[0].text
print(rating)

Overwhelmingly Positive


In [225]:
release_date = soup0.find_all('div', {'class': 'date'})[0].text.strip()
print(release_date)

5 Mar, 2026


In [226]:
developers = soup0.find_all('div', {'id': 'developers_list'})[0].text.strip()
print(developers)

Mega Crit


In [227]:
tags = []
for tag in soup0.find_all('a', {'class': 'app_tag'}):
    tags.append(tag.text.strip())
print(tags)

['Strategy', 'Roguelike', 'Card Game', 'Deckbuilding', 'Co-op', 'Roguelike Deckbuilder', 'Multiplayer', 'Turn-Based Combat', 'Card Battler', 'Indie', 'Dungeon Crawler', 'Roguelite', '2D', 'Early Access', 'Singleplayer', 'Fantasy', 'Difficult', 'Replay Value', 'Procedural Generation', 'Mouse only']


Есть проблема при входе на страницы некоторых игр, где есть возрастное ограничение и поэтому не удается с них собрать данные, если не подтвердить возраст, поэтому необходимо также через библиотеку selenium нужно выбрать возраст > 18 лет

In [228]:
def pass_age_gate(driver): #функция подтверждения возраста
    try:
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "ageYear"))) # тут крч жду до 3 секунд элемент с id ageYear и если такой элемент появился, значит перед нами форма подтверждения возраста
        logging.info("Обнаружена страница с подтверждением возраста")
        
        # тут выбираю день, месяц и год рождения, чтобы указать возраст старше 18 лет
        day = Select(driver.find_element(By.ID, "ageDay"))
        month = Select(driver.find_element(By.ID, "ageMonth"))
        year = Select(driver.find_element(By.ID, "ageYear"))
        day.select_by_visible_text("27")
        month.select_by_visible_text("September")
        year.select_by_visible_text("2005")
        
        driver.find_element(By.ID, "view_product_page_btn").click() #тут нажимаю продолжить 
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CLASS_NAME, "apphub_AppName"))) # это команда ожидания, чтобы Selenium не начал парсить страницу слишком рано
        logging.info("Возраст подтвержден, страница игры открыта")
    except TimeoutException:
        logging.info("Страница с подтверждением возраста не обнаружена")

In [232]:
def GetGame(url0):
    logging.info('Начало обработку игры')
    """
    Возвращает кортеж с url0, name, rating, release_date, developers, tags, categories

    """
    
    driver.get(url0)
    pass_age_gate(driver)
    page0 = driver.page_source
    soup0 = BeautifulSoup(page0, 'html')
    try:
        name = soup0.find_all('div', {'class': 'apphub_AppName'})[0].text.strip()
    except IndexError:
        name = None
        logging.warning('Не удалось найти название игры')

    try:
        rating = soup0.find_all('span', {'class': 'game_review_summary'})[0].text.strip()
    except IndexError:
        rating = None
        logging.warning('Не удалось найти рейтинг')
        

    try:
        release_date = pd.to_datetime(soup0.find_all('div', {'class': 'date'})[0].text.strip(), format='%d %b, %Y')
    except IndexError:
        release_date = None
        logging.warning('Не удалось найти дату релиза')

    try:
        developers = soup0.find_all('div', {'id': 'developers_list'})[0].text.strip()
    except IndexError:
        developers = None
        logging.warning('Не удалось найти разработчика')

    tags = []
    for tag in soup0.find_all('a', {'class': 'app_tag'}):
        tags.append(tag.text.strip())

    logging.info('Игра обработана')

    return url0, name, rating, release_date, developers, tags

In [233]:
games = []

for link in urls[:10]:   
    logging.info(f'Обрабатывается ссылка: {link}')
    res = GetGame(link)
    
    if res is not None:
        games.append(res)
        logging.info('Данные успешно добавлены в список games')
    else:
        logging.warning('Функция вернула None')
    
    sleep(2)
    
logging.info(f'Всего собрано игр: {len(games)}')

22:19:17 - INFO - Обрабатывается ссылка: https://store.steampowered.com/app/730/CounterStrike_2/?snr=1_7_7_230_150_1
22:19:17 - INFO - Начало обработку игры
22:19:20 - INFO - Страница с подтверждением возраста не обнаружена
22:19:20 - INFO - Игра обработана
22:19:20 - INFO - Данные успешно добавлены в список games
22:19:22 - INFO - Обрабатывается ссылка: https://store.steampowered.com/app/3321460/Crimson_Desert/?snr=1_7_7_230_150_1
22:19:22 - INFO - Начало обработку игры
22:19:26 - INFO - Страница с подтверждением возраста не обнаружена
22:19:26 - INFO - Игра обработана
22:19:26 - INFO - Данные успешно добавлены в список games
22:19:28 - INFO - Обрабатывается ссылка: https://store.steampowered.com/app/3564740/Where_Winds_Meet/?snr=1_7_7_230_150_1
22:19:28 - INFO - Начало обработку игры
22:19:31 - INFO - Страница с подтверждением возраста не обнаружена
22:19:31 - INFO - Игра обработана
22:19:31 - INFO - Данные успешно добавлены в список games
22:19:33 - INFO - Обрабатывается ссылка: htt

In [235]:
df = pd.DataFrame(games)
df.columns = ['link', 'name', 'rating', 'release_date', 'developers', 'tags']
df.head(10)

,link,name,rating,release_date,developers,tags
0,https://store.steampowered.com/app/730/Counter...,Counter-Strike 2,Mostly Positive,2012-08-21,Valve,"[FPS, Shooter, Multiplayer, Competitive, Actio..."
1,https://store.steampowered.com/app/3321460/Cri...,Crimson Desert,Very Positive,2026-03-19,Pearl Abyss,"[Open World, Action, Singleplayer, Adventure, ..."
2,https://store.steampowered.com/app/3564740/Whe...,Where Winds Meet,Mostly Positive,2025-11-14,Everstone Studio,"[Open World, Free to Play, Multiplayer, Souls-..."
3,https://store.steampowered.com/app/2868840/Sla...,Slay the Spire 2,Overwhelmingly Positive,2026-03-05,Mega Crit,"[Strategy, Roguelike, Card Game, Deckbuilding,..."
4,https://store.steampowered.com/app/252490/Rust...,Rust,Very Positive,2018-02-08,Facepunch Studios,"[Survival, Crafting, Multiplayer, Open World, ..."
5,https://store.steampowered.com/app/1808500/ARC...,ARC Raiders,Mostly Positive,2025-10-30,Embark Studios,"[Extraction Shooter, PvP, Multiplayer, PvE, Th..."
6,https://store.steampowered.com/app/1172470/Ape...,Apex Legends™,Mixed,2020-11-05,Respawn,"[Free to Play, Battle Royale, Multiplayer, FPS..."
7,https://store.steampowered.com/app/236390/War_...,War Thunder,Mostly Positive,2013-08-15,Gaijin Entertainment,"[Free to Play, Simulation, Vehicular Combat, C..."
8,https://store.steampowered.com/app/230410/Warf...,Warframe,Very Positive,2013-03-25,Digital Extremes,"[Free to Play, Looter Shooter, Action RPG, Thi..."
9,https://store.steampowered.com/app/3764200/Res...,Resident Evil Requiem,Overwhelmingly Positive,2026-02-27,"CAPCOM Co., Ltd.","[Survival Horror, Zombies, Horror, Third-Perso..."


In [236]:
!pip install openpyxl

In [237]:
df.to_excel('gamesss.xlsx', index=False)